# ORCA — Oil Spill Detection Experiments

## Objective

Train a semantic segmentation model to detect oil-spill regions in Sentinel-1 SAR imagery.

### Pipeline

Sentinel-1 SAR Image
        ↓
Preprocessing
        ↓
DARTIS `.tab` Annotation Parsing
        ↓
Pseudo-Segmentation Mask Generation
        ↓
U-Net
        ↓
Oil Probability Map
        ↓
Binary Spill Mask

## Dataset

PANGAEA Sentinel-1 SAR oil-spill dataset (DARTIS 2019).

The dataset contains:
- Sentinel-1 SAR image patches
- A tab-separated `.tab` metadata/annotation file
- Oil-spill object bounding boxes in pixel coordinates
- Oil and no-oil image patches

### Annotation format

The `DARTIS_2019.tab` file contains one row per annotated oil object. The relevant pixel-coordinate columns are `obj_patchloc_xmin`, `obj_patchloc_ymin`, `obj_patchloc_xmax`, and `obj_patchloc_ymax`. Images in the no-oil subsets have no object coordinates.

### Important limitation

The available annotations are bounding boxes rather than pixel-level segmentation masks.

Therefore, the initial U-Net baseline converts each oil bounding box into a binary rectangular pseudo-mask.

This model should therefore be treated as a baseline for the SIH prototype. Future training should use true pixel-level masks if available.

In [ ]:
import os
import random
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# =========================
# ORCA Detection Configuration
# =========================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMAGE_SIZE = 256
BATCH_SIZE = 16
NUM_EPOCHS = 30
LEARNING_RATE = 1e-3

MODEL_DIR = Path("../models/spill_detection")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = MODEL_DIR / "unet_best.pt"

print("Device:", DEVICE)
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Epochs:", NUM_EPOCHS)

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
SATELLITE_DIR = PROJECT_ROOT / "data" / "satellite"

print("Project root:")
print(PROJECT_ROOT)


print("\nSatellite directory:")
print(SATELLITE_DIR)

print("\nContents:")
for item in PROJECT_ROOT.iterdir():
    print(item)

In [ ]:
image_extensions = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
tab_files = []
image_files = []

for path in SATELLITE_DIR.rglob("*"):
    if path.is_file():
        if path.suffix.lower() in image_extensions:
            image_files.append(path)
        elif path.suffix.lower() == ".tab":
            tab_files.append(path)

print("Images:", len(image_files))
print("TAB annotation files:", len(tab_files))

if not tab_files:
    raise FileNotFoundError(f"No .tab annotation file found under {SATELLITE_DIR}")

print("\nExample images:")
for p in image_files[:10]:
    print(p)

print("\nTAB files:")
for p in tab_files:
    print(p)

In [ ]:
tab_path = next((p for p in tab_files if p.name == "DARTIS_2019.tab"), tab_files[0])

print("TAB annotation file:", tab_path)

# PANGAEA .tab files contain a metadata block followed by the tab-separated data matrix.
# Locate the actual header instead of relying on a hard-coded line number.
with tab_path.open("r", encoding="utf-8", errors="replace") as f:
    tab_lines = f.readlines()

header_idx = next(
    i for i, line in enumerate(tab_lines)
    if line.startswith("Image set") and "IMAGE (jpg_file)" in line
)

tab_df = pd.read_csv(
    tab_path,
    sep="\t",
    skiprows=header_idx,
    dtype=str,
)

print("Rows:", len(tab_df))
print("Columns:", len(tab_df.columns))
print("Unique images:", tab_df["IMAGE (jpg_file)"].nunique())

print("\nImage subsets:")
print(tab_df["Image set (subset; oc : oil/coast; ow : ...)"].value_counts(dropna=False))

print("\nTAB columns:")
for col in tab_df.columns:
    print(col)

In [ ]:
def parse_tab_annotations(tab_df):
    """
    Convert DARTIS_2019.tab object annotations into image-level bounding boxes.

    The .tab file contains one row for each annotated oil object. No-oil
    images have missing values in the object pixel-coordinate columns.

    Returns:
        dict mapping image filename -> list of dictionaries containing
        xmin, ymin, xmax, ymax
    """

    image_col = "IMAGE (jpg_file)"
    xmin_col = "Pos X [pixel] (obj_patchloc_xmin)"
    ymin_col = "Pos Y [pixel] (obj_patchloc_ymin)"
    xmax_col = "Pos X [pixel] (obj_patchloc_xmax)"
    ymax_col = "Pos Y [pixel] (obj_patchloc_ymax)"

    required_columns = [image_col, xmin_col, ymin_col, xmax_col, ymax_col]
    missing = [c for c in required_columns if c not in tab_df.columns]
    if missing:
        raise ValueError(f"Missing required TAB columns: {missing}")

    annotations = {}

    for _, row in tab_df.iterrows():
        image_name = row[image_col]
        if pd.isna(image_name):
            continue

        try:
            xmin = float(row[xmin_col])
            ymin = float(row[ymin_col])
            xmax = float(row[xmax_col])
            ymax = float(row[ymax_col])
        except (TypeError, ValueError):
            # No-oil rows have missing object coordinates.
            continue

        if xmax <= xmin or ymax <= ymin:
            continue

        annotations.setdefault(image_name, []).append({
            "xmin": xmin,
            "ymin": ymin,
            "xmax": xmax,
            "ymax": ymax,
        })

    return annotations


annotations_by_image = parse_tab_annotations(tab_df)

print("Images with oil annotations:", len(annotations_by_image))
print("Total oil objects:", sum(len(v) for v in annotations_by_image.values()))

In [ ]:
if not image_files:
    raise FileNotFoundError(f"No image files found under {SATELLITE_DIR}")

print("Example image:", image_files[0].name)

records = []

for image_path in image_files:
    boxes = annotations_by_image.get(image_path.name, [])

    records.append({
        "image": image_path,
        "annotation": boxes,
        "has_annotation": len(boxes) > 0,
    })

df = pd.DataFrame(records)

print(df.head())
print()
print(df["has_annotation"].value_counts())

In [ ]:
total_objects = 0
images_with_oil = 0
images_without_oil = 0

for _, row in df.iterrows():
    boxes = row["annotation"]

    if len(boxes) > 0:
        images_with_oil += 1
        total_objects += len(boxes)
    else:
        images_without_oil += 1

print("Total images:", len(df))
print("Images with oil:", images_with_oil)
print("Images without oil:", images_without_oil)
print("Total oil objects:", total_objects)

In [ ]:
def create_pseudo_mask(image_size, boxes):
    """
    Create a binary pseudo-segmentation mask from bounding boxes.

    image_size:
        (width, height)

    boxes:
        list of dictionaries containing bbox coordinates.
    """

    width, height = image_size

    mask = np.zeros((height, width), dtype=np.uint8)

    for box in boxes:

        xmin = max(0, int(round(box["xmin"])))
        ymin = max(0, int(round(box["ymin"])))
        xmax = min(width, int(round(box["xmax"])))
        ymax = min(height, int(round(box["ymax"])))

        if xmax <= xmin or ymax <= ymin:
            continue

        mask[ymin:ymax, xmin:xmax] = 1

    return mask

In [ ]:
sample_candidates = df[df["has_annotation"]]

if sample_candidates.empty:
    raise ValueError("No images with valid oil annotations were found in the TAB file.")

sample_row = sample_candidates.iloc[0]

image = Image.open(sample_row["image"]).convert("RGB")
boxes = sample_row["annotation"]

mask = create_pseudo_mask(image.size, boxes)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(image)
axes[0].set_title("Original SAR Image")
axes[0].axis("off")

axes[1].imshow(mask, cmap="gray")
axes[1].set_title("Pseudo Oil Mask")
axes[1].axis("off")

axes[2].imshow(image)
axes[2].imshow(mask, alpha=0.4)
axes[2].set_title("Image + Mask")
axes[2].axis("off")

plt.tight_layout()
plt.show()